In [ ]:
%pip install torch transformers numpy scipy

# Drive Integration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
import os

drive_folder_path = '/content/drive/MyDrive/HW 2'

if os.path.exists(drive_folder_path):
    %cd {drive_folder_path}
    print(f"Successfully changed directory to: {os.getcwd()}")
else:
    print(f"Error: The folder '{drive_folder_path}' does not exist. Please check the folder name and path.")

/content/drive/MyDrive/HW 2
Successfully changed directory to: /content/drive/MyDrive/HW 2


# Instead of Command Line Args

Note: LoRA needs a higher learning rate

In [ ]:
class Args:
    def __init__(self):
        self.no_cuda = False
        self.SGDR = False
        self.epochs = 30
        self.d_model = 768
        self.n_layers = 12
        self.heads = 12
        self.dropout = 0.1
        self.batchsize = 5
        self.printevery = 100
        self.lr = 0.001
        self.seqlen = 512
        self.threshold = 3
        self.savename = None
        self.loadname = None
        self.tied = 1
        self.dir_name = 'model'
        self.norm = 5.0
        self.retokenize = False
        self.verbose = False
        self.train_len = 0 # This will be set later based on data
        self.early_stopping_patience = 5
        self.early_stopping_min_delta = 0.001
        self.force_retokenize = False

opt = Args()

# Imports

In [ ]:
import os
import sys
import shutil
import random
import numpy as np
import time
import copy
import math
import pickle
import re

import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.autograd import Variable
from transformers import GPT2TokenizerFast

# Data Helper Functions (Written by Prof)

In [ ]:
def OutText(text,opt,screen=True):
    if screen:
        print(text)
    if opt.log_file:
        outFile = open(opt.log_file,"a+")
        outFile.write(text+"\n")

def read_corpus(filename,tokenizer):
    seq = []
    with open(filename,'rt') as f:
        for line in f:
            line = line.replace('\n','')
            tokens = tokenizer(line)
            for t in tokens['input_ids']:
                seq.append(t)
    return(seq)

def read_indices(filename):
    seq = []
    with open(filename,'rt') as f:
        for line in f:
            line = line.replace('\n','')
            tokens = line.split()
            for t in tokens:
                seq.append(int(t))
    return(seq)

def find_sequence_length(tokenized_data):
    max_length = 0
    for instance in tokenized_data:
        if len(instance) > max_length:
            max_length = len(instance)
    return max_length


# Transformer Code

## Embedder

This class creates $V$ word embedding vectors of size $d_{\text{model}}$

In [ ]:
class Embedder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
    def forward(self, x):
        return self.embed(x.int())

## Positional Encoding

In order to imbue the model with word's positions (which are not tracked in the attention mechanism), we add sines and cosines of specific frequencies to each dimension of the embedding vectors

In [ ]:
class PositionalEncoder(nn.Module):
    def __init__(self, d_model, max_seq_len = 4096, dropout = 0.1):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_seq_len, d_model)
        for pos in range(max_seq_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = \
                math.sin(pos / (10000 ** ((2 * i)/d_model)))
                pe[pos, i + 1] = \
                math.cos(pos / (10000 ** ((2 * (i + 1))/d_model)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x * math.sqrt(self.d_model)
        seq_len = x.size(1)
        pe = Variable(self.pe[:,:seq_len], requires_grad=False)
        pe = pe.to(x.device) # FIX: Ensure pe is on the same device as x
        x = x + pe
        return self.dropout(x)

# Norm

For normalization we map $x → \frac{x -\mu_x}{\sigma_x}=x'$. The forward function outputs $ \frac{\alpha (x - \mu_x)}{\sigma_x + \epsilon}+b$

In [ ]:
class Norm(nn.Module):
    def __init__(self, d_model, eps = 1e-6):
        super().__init__()
        self.size = d_model
        self.alpha = nn.Parameter(torch.ones(self.size))
        self.bias = nn.Parameter(torch.zeros(self.size))
        self.eps = eps

    def forward(self, x):
        norm = self.alpha * (x - x.mean(dim=-1, keepdim=True)) \
        / (x.std(dim=-1, keepdim=True) + self.eps) + self.bias
        return norm

# Attention

In [ ]:
def attention(q, k, v, d_k, mask=None, dropout=None):

    scores = torch.matmul(q, k.transpose(-2, -1)) /  math.sqrt(d_k)

    if mask is not None:
        mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask == 0, -1e9)

    scores = F.softmax(scores, dim=-1)

    if dropout is not None:
        scores = dropout(scores)

    output = torch.matmul(scores, v)
    return output


class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model, seqlen, norm, dropout = 0.1):
        super().__init__()

        self.d_model = d_model
        self.d_k = d_model // heads
        self.h = heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.register_buffer('sigma', torch.ones([seqlen, seqlen], dtype=torch.float32))
        self.norm = norm

        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):

        bs = q.size(0)

        k = self.k_linear(k).view(bs, -1, self.h, self.d_k)
        q = self.q_linear(q).view(bs, -1, self.h, self.d_k)
        v = self.v_linear(v).view(bs, -1, self.h, self.d_k)

        k = k.transpose(1,2)
        q = q.transpose(1,2)
        v = v.transpose(1,2)

        scores = attention(q, k, v, self.d_k, mask, self.dropout)

        concat = scores.transpose(1,2).contiguous().view(bs, -1, self.d_model)
        output = self.out(concat)

        return output

# Feed Forward Neural Network

This is a very simple implementation of a feed forward neural network.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout = 0.1):
        super().__init__()

        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.dropout(F.relu(self.linear_1(x)))
        x = self.linear_2(x)
        return x


def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


# Learning Rate

We use a bespoke method to cleverly modulate the learning late

In [ ]:
class CosineWithRestarts(torch.optim.lr_scheduler._LRScheduler):

    def __init__(self,
                 optimizer: torch.optim.Optimizer,
                 T_max: int,
                 eta_min: float = 0.,
                 last_epoch: int = -1,
                 factor: float = 1.) -> None:
        self.T_max = T_max
        self.eta_min = eta_min
        self.factor = factor
        self._last_restart: int = 0
        self._cycle_counter: int = 0
        self._cycle_factor: float = 1.
        self._updated_cycle_len: int = T_max
        self._initialized: bool = False
        super(CosineWithRestarts, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        if not self._initialized:
            self._initialized = True
            return self.base_lrs

        step = self.last_epoch + 1
        self._cycle_counter = step - self._last_restart

        lrs = [
            (
                self.eta_min + ((lr - self.eta_min) / 2) *
                (
                    np.cos(
                        np.pi *
                        ((self._cycle_counter) % self._updated_cycle_len) /
                        self._updated_cycle_len
                    ) + 1
                )
            ) for lr in self.base_lrs
        ]

        if self._cycle_counter % self._updated_cycle_len == 0:
            self._cycle_factor *= self.factor
            self._cycle_counter = 0
            self._updated_cycle_len = int(self._cycle_factor * self.T_max)
            self._last_restart = step

        return lrs


# Decoder Code

In [ ]:
class DecoderLayerGPT(nn.Module):
    def __init__(self, d_model, heads, seqlen, norm, dropout=0.1):
        super().__init__()
        self.norm_1 = Norm(d_model)
        self.norm_2 = Norm(d_model)

        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

        self.attn_1 = MultiHeadAttention(heads, d_model, seqlen, norm, dropout=dropout)
        self.ff = FeedForward(d_model, dropout=dropout)

    def forward(self, x, mask):
        x2 = self.norm_1(x)
        x = x + self.dropout_1(self.attn_1(x2, x2, x2, mask))
        x2 = self.norm_2(x)
        x = x + self.dropout_2(self.ff(x2))
        return x

# Decoder Stack

## Remember that GPT2 is Decoder only


In [ ]:
class DecoderGPT(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads, seqlen, norm, dropout):
        super().__init__()
        self.N = N
        self.embed = Embedder(vocab_size, d_model)
        self.pe = PositionalEncoder(d_model, dropout=dropout)
        self.layers = get_clones(DecoderLayerGPT(d_model, heads, seqlen, norm, dropout), N)
        self.norm = Norm(d_model)
    def forward(self, trg, mask):
        x = self.embed(trg)
        x = self.pe(x)
        for i in range(self.N):
            x = self.layers[i](x, mask)
        return self.norm(x)

# Putting it All Together: The Tranformer

In [ ]:
class TransformerGPT(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads, dropout,opt):
        super().__init__()
        self.decoder = DecoderGPT(vocab_size, d_model, N, heads, opt.seqlen, opt.norm, dropout)
        self.out = nn.Linear(d_model, vocab_size)
        self.opt = opt
        self.vocab_size = vocab_size
        self.register_buffer('indices', torch.arange(vocab_size, dtype=torch.long)) # FIX: Registers indices as a buffer

    def forward(self, trg, trg_mask):
        d_output = self.decoder(trg, trg_mask)
        if self.opt.tied == 0:
            output = self.out(d_output)
        else:
            output = torch.matmul(d_output,self.decoder.embed.embed(self.indices).transpose(0,1)) # FIX: Uses self.indices
        return output

def get_modelGPT(opt, vocab_size):
    import torch.nn as nn
    assert opt.d_model % opt.heads == 0
    assert opt.dropout < 1
    model = TransformerGPT(vocab_size, opt.d_model, opt.n_layers, opt.heads, opt.dropout, opt)
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    if hasattr(opt, "device"):
        model = model.to(opt.device)
    return model


In [ ]:
import torch

class LoRALinearLayer(nn.Module):
  def __init__(self, d, k, rank):
    super(LoRALinearLayer, self).__init__()

    self.r = rank
    self.numInputFeatures = d
    self.numOutputFeatures = k

    self.A_matrix = nn.Parameter(torch.Tensor(rank, k))
    self.B_matrix = nn.Parameter(torch.Tensor(d, rank))

    nn.init.kaiming_uniform_(self.A_matrix, a=math.sqrt(5))
    nn.init.zeros_(self.B_matrix)

  def forward(self, x):
    return x @ self.B_matrix @ self.A_matrix


In [ ]:
class LoRA(nn.Module):
  def __init__(self, original_linear_layer, rank):
    super(LoRA, self).__init__()

    self.original_linear_layer = original_linear_layer

    self.original_linear_layer.requires_grad = False

    self.lora_wts = LoRALinearLayer(self.original_linear_layer.in_features, self.original_linear_layer.out_features, rank)

  def forward(self, x):
    return self.original_linear_layer(x) + self.lora_wts(x)

In [ ]:
def apply_lora(model, r=8, modules=("q_linear", "k_linear")):
  for param in model.parameters():
    param.requires_grad = False

  for name, module in model.named_modules():
        if any(target_name in name for target_name in modules):
            if isinstance(module, nn.Linear):
                parent_name = '.'.join(name.split('.')[:-1])
                child_name = name.split('.')[-1]
                parent_module = model.get_submodule(parent_name)

                lora_layer = LoRA(module, rank=r)
                setattr(parent_module, child_name, lora_layer)

  return model


# Additional Helpers

In [ ]:
import torch

def load_ckpt_allowing_pad_growth(model, state, verbose=True):
    """
    Load 'state' into 'model' by:
      - expanding checkpoint tensors that are off by +1 on the first dim (vocab) for
        decoder.embed.embed.weight, out.weight, out.bias (plus a few common aliases)
      - dropping any keys whose shapes still don't match after that
      - using strict=False so missing/unexpected keys are ignored

    This is a helper that allows us to add the special tokens and guarantees no
    size-mismatch errors from load_state_dict.
    """
    model_sd = model.state_dict()
    adj = {}  # only put tensors that we know match the model

    def can_expand_vocab_dim(ckpt_t, model_t):
      return (
          ckpt_t.dim() == model_t.dim()
          and ckpt_t.shape[1:] == model_t.shape[1:]
          and model_t.shape[0] >= ckpt_t.shape[0]   # allow +N pad tokens
      )

    def expand_first_dim(ckpt_t, target_shape, pad_id=None):
        out = ckpt_t.new_zeros(target_shape)
        rows = min(ckpt_t.size(0), out.size(0))
        out[:rows].copy_(ckpt_t[:rows])
        # Optionally zero the pad row
        if pad_id is not None and pad_id < target_shape[0]:
            out[pad_id].zero_()
        return out

    # vocab-sized param names to consider for expansion
    mat_vocab_keys = {
        "decoder.embed.embed.weight",
        "tok_embed.weight", "embedding.weight", "embed_tokens.weight",
        "out.weight", "proj_out.weight", "lm_head.weight",
    }
    vec_vocab_keys = {
        "out.bias", "proj_out.bias", "lm_head.bias",
    }

    for k, v in state.items():
        if k not in model_sd:
            # not used by this model -> ignore
            continue

        tgt = model_sd[k]
        if v.shape == tgt.shape:
            adj[k] = v  # perfect match
            continue

        # Try vocab +1 growth path
        if (k in mat_vocab_keys or k in vec_vocab_keys) and can_expand_vocab_dim(v, tgt):
            adj[k] = expand_first_dim(v, tgt.shape, pad_id=opt.pad_token_id)
            if verbose:
                print(f"[pad+1] expanded {k}: {tuple(v.shape)} -> {tuple(tgt.shape)}")
            continue

        # Otherwise, shapes differ: skip this key (keep model init)
        if verbose:
            print(f"[skip] {k}: ckpt {tuple(v.shape)} vs model {tuple(tgt.shape)}")

    # Now load only the adjusted, shape-safe subset
    missing, unexpected = model.load_state_dict(adj, strict=False)
    if verbose:
        print("non-strict load -> missing (kept model init):",
              missing[:6], "..." if len(missing) > 6 else "")
        print("non-strict load -> unexpected (ignored):",
              unexpected[:6], "..." if len(unexpected) > 6 else "")


def to_device_batch(batch, device):
    # Handles dicts, tuples, or tensors
    if isinstance(batch, dict):
        out = {}
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                if k == "input_ids":
                    out[k] = v.to(device=device, dtype=torch.long, non_blocking=True)
                else:
                    out[k] = v.to(device=device, non_blocking=True)
            else:
                out[k] = v
        return out
    elif isinstance(batch, (tuple, list)):
        return [to_device_batch(x, device) for x in batch]
    elif isinstance(batch, torch.Tensor):
        return batch.to(device=device, non_blocking=True)
    else:
        return batch


In [ ]:
from transformers import GPT2TokenizerFast
import json
import hashlib

def get_tokenizer_cache_dir():
    return os.path.join("saved", "tokenizer")

def load_or_build_tokenizer(opt, force_retokenize=False):
    """
    Load tokenizer from cache if it exists, otherwise build from GPT-2,
    add pad + custom tokens, then save it to disk.
    """
    cache_dir = get_tokenizer_cache_dir()

    # If cache exists and we are not forcing a rebuild → just load it
    if os.path.isdir(cache_dir) and not force_retokenize:
        print(f"Loading cached tokenizer from {cache_dir}")
        tokenizer = GPT2TokenizerFast.from_pretrained(
            cache_dir,
            model_max_length=1_000_000
        )
        return tokenizer

    # Otherwise build from base GPT-2
    print("Building tokenizer from base GPT-2")
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2", model_max_length=1_000_000)

    # 1) Ensure pad token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

    custom_tokens = ["[A]", "[B]", "[C]", "[D]"]
    added = tokenizer.add_tokens(custom_tokens, special_tokens=False)
    print(f"Added {added} custom tokens; new vocab size = {len(tokenizer)}")

    os.makedirs(cache_dir, exist_ok=True)
    tokenizer.save_pretrained(cache_dir)
    print(f"Saved tokenizer with vocab_size={len(tokenizer)} to {cache_dir}")

    return tokenizer

# Training Code (Used on Wikipedia Dataset)

In [ ]:
def train_model(model, opt):
    print("training model...")
    model.train()
    count = 0

    aa = opt.seqlen
    bb = opt.batchsize
    opt.bb = bb
    offsets = []
    stride = max(aa, int(len(opt.train) / max(bb, 1)))  # avoid 0/too-small stride
    for i in range(0, len(opt.train), stride):
        offsets.append(i)
    print('stride = ', stride)
    print('offsets = ', offsets)

    nopeak_mask = np.triu(np.ones((bb, aa, aa), dtype=np.int32), k=1)
    mask = (torch.from_numpy(nopeak_mask) == 0).to(opt.device)

    pad_id = getattr(opt, "trg_pad", None)

    best_val_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(opt.epochs):
        start = time.time()
        total_loss = 0.0
        total = 0

        for i in range(0, int((stride - aa) / 1), aa):
            trg = torch.zeros((bb, aa), dtype=torch.long)
            for j in range(aa):
                for k in range(bb):
                    trg[k, j] = opt.train[offsets[k] + i + j]
            trg = trg.to(device=opt.device, dtype=torch.long)

            preds = model(trg, mask)  # [B, L, V]
            preds = preds[:, :-1, :].contiguous().view(-1, preds.size(2))
            ys = trg[:, 1:].contiguous().view(-1)

            if pad_id is not None:
                pad_mask = (trg[:, 1:] == pad_id).view(-1)
                ys = ys.clone()
                ys[pad_mask] = -100

            opt.optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(preds, ys)
            loss.backward()
            opt.optimizer.step()
            if opt.SGDR:
                opt.sched.step()

            total_loss += loss.item()
            total += 1

            count += 1
            if count % max(int(opt.printevery / max(bb, 1)), 1) == 0:
                count = 0
                p = int(100 * (i * bb + 1) / max(len(opt.train), 1))
                avg_loss = total_loss / max(total, 1)
                text = (
                    f"   {int((time.time()-start)//60)}m: epoch {epoch+1} "
                    f"[{'#'*(p//5)}{' '*(20-(p//5))}]  {p}%  "
                    f"wps = {float(aa*bb*total)/max((time.time()-start),1e-6):7.0f} "
                    f"loss = {avg_loss:.3f} {math.exp(avg_loss):7.1f}"
                )
                OutText(text, opt)

        if opt.savename is not None:
            print('saving weights...')
            torch.save(model.state_dict(), opt.savename + '/model_weights')

        avg_loss = total_loss / max(total, 1)
        wps = float(aa * bb * total) / max((time.time() - start), 1e-6)
        text = (
            f"{int((time.time()-start)//60)}m: epoch {epoch+1} "
            f"[{'#'*(100//5)}{' '*(20-(100//5))}]  100%  loss = {avg_loss:.3f}\n"
            f"epoch {epoch+1} complete, wps = {wps:7.0f} loss = {avg_loss:.03f} "
            f"ppl = {math.exp(avg_loss):7.1f}"
        )
        OutText(text, opt)

        # Evaluate on validation set
        val_loss = test_model(model, opt, epoch)

        # Early stopping logic
        if val_loss < best_val_loss - opt.early_stopping_min_delta:
            best_val_loss = val_loss
            epochs_no_improve = 0
            # Optionally save best model
            if opt.savename is not None:
                print('saving best model weights...')
                torch.save(model.state_dict(), opt.savename + '/best_model_weights')
        else:
            epochs_no_improve += 1
            OutText(f"No improvement for {epochs_no_improve} epochs. Best loss: {best_val_loss:.3f}", opt)
            if epochs_no_improve >= opt.early_stopping_patience:
                OutText(f"Early stopping triggered after {epoch+1} epochs.", opt)
                break # Exit training loop

# Model Testing Code (Used on Wiki Dataset)

In [ ]:
def test_model(model, opt, epoch):
    model.eval()
    start = time.time()

    aa = opt.seqlen
    bb = 1
    opt.bb = bb

    nopeak_mask = np.triu(np.ones((bb, aa, aa), dtype=np.int32), k=1)
    mask = torch.from_numpy(nopeak_mask) == 0
    mask = mask.to(device=opt.device)  # bool tensor

    count = 0
    total_loss = 0.0
    pad_id = getattr(opt, "trg_pad", None)

    for i in range(0, len(opt.valid) - 2 * aa, aa): # Use opt.valid for evaluation
        count += 1

        trg = torch.zeros((bb, aa), dtype=torch.long)
        for j in range(aa):
            for k in range(bb):
                trg[k, j] = opt.valid[i + j] # Use opt.valid data

        trg = trg.to(device=opt.device, dtype=torch.long)

        preds = model(trg, mask)               # [B, L, V]
        preds = preds[:, :-1, :].contiguous().view(-1, preds.size(2))
        ys = trg[:, 1:].contiguous().view(-1)

        if pad_id is not None:
            pad_mask = (trg[:, 1:] == pad_id).view(-1)
            ys = ys.clone() # Ensure clone for modification
            ys[pad_mask] = -100  # ignore_index

        loss = F.cross_entropy(preds.reshape(-1, preds.size(-1)), ys)
        total_loss += loss.item()

    OutText(' ', opt)
    avg_loss = total_loss / max(count, 1)
    ppl = math.exp(avg_loss)
    text = (
        f"{int((time.time()-start)//60)}m: VALID {epoch+1} " # Changed to VALID
        f"[{'#'*(100//5)}{' '*(20-(100//5))}]  100%  loss = {avg_loss:.3f}\n"
        f"epoch {epoch+1} complete, loss = {avg_loss:.03f} ppl = {ppl:7.1f}"
    )
    OutText(text, opt)
    OutText(' ', opt)

    model.train()
    opt.bb = opt.batchsize
    return avg_loss


# Validation Code

In [ ]:
def validate_obqa_model(model, opt, tokenizer, epoch):
    model.eval()
    start = time.time()

    data = []
    with open('obqa.valid.txt', 'rt') as f:
        for line in f:
            line = line.rstrip('\n')
            tokens = line.split('|')
            d = {
                'fact': tokens[0],
                'stem': tokens[1],
                'A': tokens[2],
                'B': tokens[3],
                'C': tokens[4],
                'D': tokens[5],
                'Answer': tokens[6],
            }
            data.append(d)

    encoded_data_val = []
    for instance in data:
        fact = instance['fact']
        stem = instance['stem']
        A = instance['A']; B = instance['B']; C = instance['C']; D = instance['D']
        Answer = instance['Answer']
        encoding_line = f"[START] {fact} {stem} A: {A} B: {B} C: {C} D: {D} [ANSWER] {Answer}"
        ids = tokenizer(encoding_line)['input_ids']
        encoded_data_val.append(ids)

    pad_id = tokenizer.pad_token_id
    max_length_val = find_sequence_length(encoded_data_val)
    print(f"Validation max_length: {max_length_val}")

    for seq in encoded_data_val:
        if len(seq) < max_length_val:
            seq.extend([pad_id] * (max_length_val - len(seq)))

    encoded_data_val = torch.tensor(encoded_data_val, dtype=torch.long).flatten()

    # Use actual data length, not opt.seqlen
    aa = max_length_val
    bb = opt.batchsize # Use batchsize for validation as well
    total_len_val = len(encoded_data_val)
    stride_val = max(aa, int(total_len_val / max(bb, 1)))
    offsets_val = list(range(0, total_len_val, stride_val))

    nopeak_mask = np.triu(np.ones((bb, aa, aa), dtype=np.int32), k=1)
    mask = (torch.from_numpy(nopeak_mask) == 0).to(opt.device)

    total_loss = 0.0
    total_batches = 0

    upper_val = int((stride_val - aa) / 1)
    for i in range(0, max(upper_val, 0), aa):
        trg = torch.zeros((bb, aa), dtype=torch.long)
        for j in range(aa):
            for k in range(bb):
                idx = offsets_val[k] + i + j
                if idx < total_len_val:
                    trg[k, j] = encoded_data_val[idx]
                else:
                    trg[k, j] = pad_id
        trg = trg.to(device=opt.device, dtype=torch.long)

        preds = model(trg, mask)  # [B, L, V]



        preds = preds[:, :-1, :].contiguous().view(-1, preds.size(2))
        ys = trg[:, 1:].contiguous().view(-1)

        pad_mask = (trg[:, 1:] == pad_id).view(-1)
        ys_loss = ys.clone()
        ys_loss[pad_mask] = -100

        loss = F.cross_entropy(preds, ys_loss)
        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    ppl = math.exp(avg_loss)
    OutText(f"\nVALIDATION Epoch {epoch+1}: loss = {avg_loss:.3f} ppl = {ppl:7.1f}", opt)
    model.train()
    opt.bb = opt.batchsize
    return avg_loss


# Finetuner Code (The Big One)

In [ ]:
def finetune_obqa(model, opt, tokenizer):
    print("finetuning....")
    model.train()

    #  Load data
    data = []
    with open("obqa.train.txt", "rt") as f:
        for line in f:
            line = line.replace("\n", "")
            tokens = line.split("|")
            d = {
                "fact": tokens[0],
                "stem": tokens[1],
                "A": tokens[2],
                "B": tokens[3],
                "C": tokens[4],
                "D": tokens[5],
                "Answer": tokens[6],
            }
            data.append(d)
    print("data: %d" % (len(data)))

    # Encode
    encoded_data = []
    for instance in data:
        fact = instance["fact"]
        stem = instance["stem"]
        A = instance["A"]
        B = instance["B"]
        C = instance["C"]
        D = instance["D"]
        Answer = instance["Answer"]
        encoding_line = f"[START] {fact} {stem} A: {A} B: {B} C: {C} D: {D} [ANSWER] {Answer}"
        ids = tokenizer(encoding_line)["input_ids"]
        encoded_data.append(ids)

    max_length = find_sequence_length(encoded_data)
    print("max_length: ", max_length)

    # Pad to max length and keep as 2D tensor
    pad_id = tokenizer.pad_token_id
    for seq in encoded_data:
        if len(seq) < max_length:
            seq.extend([pad_id] * (max_length - len(seq)))

    # Convert to [N, L] tensor
    encoded_tensor = torch.tensor(encoded_data, dtype=torch.long)
    num_examples = encoded_tensor.size(0)
    seq_len = encoded_tensor.size(1)

    # Batching setup
    bb = opt.batchsize
    opt.bb = bb

    # Shuffle data for training
    indices = torch.randperm(num_examples)
    encoded_tensor = encoded_tensor[indices]

    print(f"Training data shape: {encoded_tensor.shape}")

    # Early Stopping variables
    best_val_loss = float("inf")
    epochs_no_improve = 0

    accum_steps = getattr(opt, "gradient_accumulation_steps", 1)

    for epoch in range(opt.epochs):
        start = time.time()
        total_loss = 0.0
        total = 0
        count = 0

        # Iterate in batches
        for i in range(0, num_examples, bb):
            # Get batch
            trg = encoded_tensor[i : i + bb].to(opt.device)
            curr_bs = trg.size(0)

            # Create mask for this specific batch size
            nopeak_mask = np.triu(np.ones((curr_bs, seq_len, seq_len), dtype=np.int32), k=1)
            mask = (torch.from_numpy(nopeak_mask) == 0).to(opt.device)

            # Forward pass
            logits = model(trg, mask)                        # [B, L, V]
            logits_next = logits[:, :-1, :]                  # [B, L-1, V]

            # LM loss
            ys_tokens = trg[:, 1:]                           # [B, L-1]
            pad_mask  = (ys_tokens == pad_id)                # [B, L-1]
            ys_for_lm = ys_tokens.masked_fill(pad_mask, -100)

            lm_loss = F.cross_entropy(
                logits_next.reshape(-1, logits_next.size(-1)),
                ys_for_lm.reshape(-1)
            )

            # Classification loss at [ANSWER]
            ANSWER_ID = tokenizer.encode("[ANSWER]", add_special_tokens=False)[0]
            cand_ids  = torch.tensor([
                tokenizer.encode(" A", add_special_tokens=False)[0],
                tokenizer.encode(" B", add_special_tokens=False)[0],
                tokenizer.encode(" C", add_special_tokens=False)[0],
                tokenizer.encode(" D", add_special_tokens=False)[0],
            ], device=trg.device, dtype=torch.long)

            is_answer = (trg == ANSWER_ID)
            ans_cnt   = is_answer.sum(dim=1)
            ans_pos   = is_answer.float().argmax(dim=1).long()

            valid_rows = (ans_cnt == 1) & (ans_pos < (trg.size(1) - 1))
            cls_loss = torch.zeros((), device=trg.device)

            if valid_rows.any():
                keep = valid_rows.nonzero(as_tuple=False).flatten()
                row_idx = keep
                pos_idx = ans_pos.index_select(0, keep)

                gold_token_ids = trg[row_idx, pos_idx + 1]

                gold_class = torch.full_like(gold_token_ids, -1)
                for j in range(4):
                    gold_class = torch.where(
                        gold_token_ids == cand_ids[j],
                        torch.tensor(j, device=trg.device, dtype=gold_class.dtype),
                        gold_class
                    )

                keep2 = (gold_class >= 0)
                if keep2.any():
                    row_idx2 = row_idx.index_select(0, keep2.nonzero(as_tuple=False).flatten())
                    pos_idx2 = pos_idx.index_select(0, keep2.nonzero(as_tuple=False).flatten())
                    gold_cls2 = gold_class.index_select(0, keep2.nonzero(as_tuple=False).flatten())

                    ans_logits = logits_next[row_idx2, pos_idx2, :]
                    logits_4 = ans_logits.index_select(1, cand_ids)
                    cls_loss = F.cross_entropy(logits_4, gold_cls2)

            alpha = 1.0
            loss = lm_loss + alpha * cls_loss

            # Backward
            loss.backward()

            if (total + 1) % accum_steps == 0:
                opt.optimizer.step()
                opt.optimizer.zero_grad(set_to_none=True)
                if opt.SGDR:
                    opt.sched.step()

            total_loss += loss.item() * accum_steps
            total += 1
            count += 1

            if count % max(int(opt.printevery / max(bb, 1)), 1) == 0:
                p = int(100 * (i + bb) / num_examples)
                avg_loss = total_loss / max(total, 1)
                wps = float(seq_len * bb * total) / max((time.time() - start), 1e-6)
                OutText(f"   {int((time.time()-start)//60)}m: Finetuning epoch {epoch+1} "
                        f"[{'#'*(p//5)}{' '*(20-(p//5))}]  {p}%  wps = {wps:7.0f} "
                        f"loss = {avg_loss:.3f} {math.exp(min(avg_loss, 100)):7.1f}", opt)

        # End of epoch
        avg_loss = total_loss / max(total, 1)
        wps = float(seq_len * bb * total) / max((time.time() - start), 1e-6)
        OutText(f"{int((time.time()-start)//60)}m: Finetuning epoch {epoch+1} "
                f"[{'#'*(100//5)}{' '*(20-(100//5))}]  100%  loss = {avg_loss:.3f}\n"
                f"epoch {epoch+1} complete, wps = {wps:7.0f} loss = {avg_loss:.03f} "
                f"ppl = {math.exp(min(avg_loss, 100)):7.1f}", opt)

        if opt.savename is not None:
            torch.save(model.state_dict(), opt.savename + "/finetuned_model_weights")

        val_loss = validate_obqa_model(model, opt, tokenizer, epoch)

        if val_loss < best_val_loss - opt.early_stopping_min_delta:
            best_val_loss = val_loss
            epochs_no_improve = 0
            if opt.savename is not None:
                torch.save(model.state_dict(), opt.savename + "/best_finetuned_model_weights")
        else:
            epochs_no_improve += 1
            OutText(f"No improvement on validation for {epochs_no_improve} epochs. Best loss: {best_val_loss:.3f}", opt)
            if epochs_no_improve >= opt.early_stopping_patience:
                OutText(f"Early stopping triggered after {epoch+1} epochs.", opt)
                break

# Test Accuracy of Finetuned Model on OBQA

In [ ]:
def testmodel_obqa(model, opt, tokenizer, print_examples=False):
    import numpy as np
    import torch
    import torch.nn.functional as F
    import time
    from collections import Counter # Import Counter for distribution

    print('validating....')

    #  Load data
    data = []
    with open('obqa.test.txt','rt') as f:
        for line in f:
            line = line.rstrip('\n')
            tokens = line.split('|')
            data.append({
                'fact': tokens[0],
                'stem': tokens[1],
                'A': tokens[2],
                'B': tokens[3],
                'C': tokens[4],
                'D': tokens[5],
                'Answer': tokens[6].strip(),
            })
    for i in range(min(5, len(data))):
        print(i, data[i])
    print('data: %d' % (len(data)))

    model.eval()

    # answer token ids (space-prefixed)
    A_id = tokenizer.encode(' A', add_special_tokens=False)[0]
    B_id = tokenizer.encode(' B', add_special_tokens=False)[0]
    C_id = tokenizer.encode(' C', add_special_tokens=False)[0]
    D_id = tokenizer.encode(' D', add_special_tokens=False)[0]
    cand_ids = torch.tensor([A_id, B_id, C_id, D_id], dtype=torch.long)

    answer_tag_id = tokenizer.encode('[ANSWER]', add_special_tokens=False)[0]
    pad_id = tokenizer.pad_token_id
    # seqlen = opt.seqlen
    bs = max(1, getattr(opt, 'batchsize', 1))

    def build_prompt_no_label(ex):
        # used for prediction at [ANSWER]
        return f"[START] {ex['fact']} {ex['stem']} A: {ex['A']} B: {ex['B']} C: {ex['C']} D: {ex['D']} [ANSWER]"

    def build_full_with_label(ex):
        # used for token-level loss (optional)
        return f"[START] {ex['fact']} {ex['stem']} A: {ex['A']} B: {ex['B']} C: {ex['C']} D: {ex['D']} [ANSWER] {ex['Answer']}"

    # tokenize all at once to avoid Python overhead
    prompts = [build_prompt_no_label(ex) for ex in data]
    full_prompts = [build_full_with_label(ex) for ex in data]

    enc_prompts = tokenizer(prompts, padding=False, truncation=False)
    enc_full    = tokenizer(full_prompts, padding=False, truncation=False)

    encoded_test = []
    for ex in data:
        line = f"[START] {ex['fact']} {ex['stem']} A: {ex['A']} B: {ex['B']} C: {ex['C']} D: {ex['D']} [ANSWER]"
        ids = tokenizer(line)['input_ids']
        encoded_test.append(ids)

    # Calculate actual max length (like in training!)
    max_length = find_sequence_length(encoded_test)
    print(f"Test max_length: {max_length}")

    seqlen = max_length  # Use actual data length!


    # helper: pad/truncate to seqlen and record [ANSWER] position
    def pad_trunc_and_answer_pos(batch_input_ids):
        batch_ids = []
        batch_anspos = []
        for ids in batch_input_ids:
            # find last [ANSWER] just in case
            ids_t = torch.tensor(ids, dtype=torch.long)
            ans_pos = (ids_t == answer_tag_id).nonzero(as_tuple=True)[0]
            if ans_pos.numel() == 0:
                # if somehow missing, put it at last token before truncation
                ans_idx = min(len(ids) - 1, seqlen - 2)
            else:
                ans_idx = int(ans_pos[-1].item())

            # truncate/pad
            if len(ids) > seqlen:
                ids_cut = ids[:seqlen]
                # if [ANSWER] got cut off, back off to the last valid spot
                if ans_idx >= seqlen:
                    ans_idx = seqlen - 2
            else:
                ids_cut = ids + [pad_id] * (seqlen - len(ids))

            batch_ids.append(torch.tensor(ids_cut, dtype=torch.long))
            batch_anspos.append(ans_idx)
        return torch.stack(batch_ids, dim=0), torch.tensor(batch_anspos, dtype=torch.long)

    X_pred, ans_pos = pad_trunc_and_answer_pos(enc_prompts['input_ids'])
    X_loss, _       = pad_trunc_and_answer_pos(enc_full['input_ids'])

    device = opt.device
    cand_ids = cand_ids.to(device)

    # batching
    def batches(X, anspos=None, B=bs):
        N = X.size(0)
        for s in range(0, N, B):
            e = min(N, s+B)
            if anspos is None:
                yield X[s:e], None
            else:
                yield X[s:e], anspos[s:e]

    # metrics
    start = time.time()
    exact_correct = 0
    exact_total = 0
    token_correct = 0
    token_total = 0
    loss_sum = 0.0
    loss_den = 0

    predicted_choices = [] # Store predicted choices here

    # setup once
    ANSWER_ID = tokenizer.encode("[ANSWER]", add_special_tokens=False)[0]
    cand_ids = torch.tensor([
        tokenizer.encode(" A", add_special_tokens=False)[0],
        tokenizer.encode(" B", add_special_tokens=False)[0],
        tokenizer.encode(" C", add_special_tokens=False)[0],
        tokenizer.encode(" D", add_special_tokens=False)[0],
    ], dtype=torch.long, device=device)

    # Exact-match on prompts WITHOUT label
    with torch.no_grad():
        # candidate letter token-ids (space-prefixed)
        cand_ids = torch.tensor([
            tokenizer.encode(" A", add_special_tokens=False)[0],
            tokenizer.encode(" B", add_special_tokens=False)[0],
            tokenizer.encode(" C", add_special_tokens=False)[0],
            tokenizer.encode(" D", add_special_tokens=False)[0],
        ], device=device, dtype=torch.long)

        N = X_pred.size(0)
        examples_printed = 0
        if print_examples:
            print("\n--- Model Prediction Examples ---")
        for s in range(0, N, bs):
            e  = min(N, s + bs)
            xb = X_pred[s:e].to(device)          # [B, L]
            ap = ans_pos[s:e].to(device)         # [B] (position of [ANSWER])
            B, L = xb.shape

            # guard: we need to predict token at t+1, so ap < L-1
            valid = (ap < (L - 1))
            if not valid.any():
                continue

            xb_valid = xb[valid]                       # [K, L]
            ap_valid = ap[valid]                       # [K]
            K  = xb_valid.size(0)

            # causal mask [K, L, L]
            np_mask = np.triu(np.ones((K, L, L), dtype=np.int32), k=1)
            mask    = (torch.from_numpy(np_mask) == 0).to(device)

            logits = model(xb_valid, mask)             # [K, L, V]
            logits_next = logits[:, :-1, :]      # [K, L-1, V]

            row = torch.arange(K, device=device) # [K]
            ans_logits = logits_next[row, ap_valid, :] # [K, V]   (distribution over token after [ANSWER])
            logits_4   = ans_logits.index_select(1, cand_ids)  # [K, 4]
            pred_idx   = logits_4.argmax(1)      # [K] in {0,1,2,3}

            # record predicted choices
            for p in pred_idx:
                predicted_choices.append(['A', 'B', 'C', 'D'][p.item()])

            # gold 0..3 from dataset letters aligned to this batch slice, then filter by 'valid'
            gold_all = torch.tensor(
                ["ABCD".index(ex['Answer']) for ex in data[s:e]],
                device=device, dtype=torch.long
            )
            gold = gold_all[valid]               # [K]

            exact_correct += int((pred_idx == gold).sum().item())
            exact_total   += int(gold.numel())

            if print_examples and examples_printed < 8: # Print only up to 5 examples
                for i_batch in range(K):
                    if examples_printed >= 5: break
                    original_idx = s + valid.nonzero(as_tuple=False).flatten()[i_batch].item()
                    predicted_char = ['A', 'B', 'C', 'D'][pred_idx[i_batch].item()]
                    true_char = data[original_idx]['Answer']
                    prompt_text = prompts[original_idx]

                    print(f"\nExample {examples_printed + 1} (from test set index {original_idx}):")
                    print(f"  Prompt: {prompt_text}")
                    print(f"  Predicted Answer: {predicted_char}")
                    print(f"  True Answer: {true_char}")
                    examples_printed += 1

    # Print distribution of predicted choices
    choice_distribution = Counter(predicted_choices)
    print("\n--- Predicted Choice Distribution ---")
    for choice, count in choice_distribution.most_common():
        print(f"  {choice}: {count} ({count/len(predicted_choices):.2%})")

    # Token-level loss/accuracy
    with torch.no_grad():
        for xb, _ in batches(X_loss, None, bs):
            xb = xb.to(device)
            B, L = xb.shape
            np_mask = np.triu(np.ones((B, L, L), dtype=np.int32), k=1)
            mask = (torch.from_numpy(np_mask) == 0).to(device)

            logits = model(xb, mask)             # [B, L, V]
            # predict next token
            logits_next = logits[:, :-1, :]      # [B, L-1, V]
            ys = xb[:, 1:]                       # [B, L-1]

            # ignore pads in loss/acc
            pad_mask = (ys == pad_id)
            ys_for_loss = ys.clone()
            ys_for_loss[pad_mask] = -100

            loss = F.cross_entropy(
                logits_next.reshape(-1, logits_next.size(-1)),
                ys_for_loss.reshape(-1),
                reduction='sum'
            )
            loss_sum += float(loss.item())
            loss_den += int((ys_for_loss != -100).sum().item())

            preds_ids = logits_next.argmax(dim=-1)  # [B, L-1]
            correct = (preds_ids == ys) & (~pad_mask)
            token_correct += int(correct.sum().item())
            token_total   += int((~pad_mask).sum().item())

    exact_acc = exact_correct / max(1, exact_total)
    token_acc = token_correct / max(1, token_total)
    avg_loss  = loss_sum / max(1, loss_den)

    OutText(f"{int((time.time()-start)//60)}m: TEST complete, loss = {avg_loss:.03f} "
            f"token_acc = {token_acc:.3f} exact_match = {exact_acc:.3f}", opt)

    model.train()
    return exact_acc

In [ ]:
def setup_model_and_data(model, tokenizer, opt):
    # === Special tokens + resize (same idea as before, kept for context) ===
    print("\n=== Adding special tokens and resizing model (idempotent) ===")

    new_special_tokens = ["[START]", "[ANSWER]", "[CLS]", "[SEP]", "[END]"]

    # Only add tokens that aren't already there
    existing_specials = list(getattr(tokenizer, "additional_special_tokens", []))
    tokens_to_add = [t for t in new_special_tokens if t not in existing_specials]

    if tokens_to_add:
        updated_specials = existing_specials + tokens_to_add
        num_added = tokenizer.add_special_tokens(
            {"additional_special_tokens": updated_specials}
        )
        print(f"Added {num_added} new special tokens")
    else:
        num_added = 0
        print("All OBQA special tokens already present; no new tokens added.")

    print("Current additional special tokens:", tokenizer.additional_special_tokens)

    for text in [' A', ' B', ' C', ' D']:
        _ = tokenizer(text)

    new_vocab_size = len(tokenizer)
    old_vocab = model.decoder.embed.embed.weight.size(0)
    print(f"Tokenizer vocab size: {new_vocab_size}, model vocab size: {old_vocab}")

    # Only resize if vocab actually grew
    if new_vocab_size > old_vocab:
        print("Resizing embeddings and output layer (memory-efficient)...")

        # Move old weights to CPU
        old_embed_weight = model.decoder.embed.embed.weight.data.cpu()
        if opt.tied == 0:
            old_out_weight = model.out.weight.data.cpu()
            old_out_bias = model.out.bias.data.cpu()

        # Delete old layers to free GPU memory
        del model.decoder.embed.embed
        if opt.tied == 0:
            del model.out
        if opt.device.type == "cuda":
            torch.cuda.empty_cache()

        # New embedding on CPU
        new_embedding = nn.Embedding(new_vocab_size, opt.d_model)
        with torch.no_grad():
            new_embedding.weight[:old_vocab].copy_(old_embed_weight)

        # New output layer on CPU
        if opt.tied == 0:
            new_out = nn.Linear(opt.d_model, new_vocab_size)
            with torch.no_grad():
                new_out.weight[:old_vocab].copy_(old_out_weight)
                new_out.bias[:old_vocab].copy_(old_out_bias)

        # Assign new layers
        model.decoder.embed.embed = new_embedding
        if opt.tied == 0:
            model.out = new_out

        # Update indices buffer
        model.register_buffer('indices', torch.arange(new_vocab_size, dtype=torch.long))

        # Cleanup
        del old_embed_weight
        if opt.tied == 0:
            del old_out_weight, old_out_bias

        opt.vocab_size = new_vocab_size
        opt.pad_token_id = tokenizer.pad_token_id

        print(f"Model resized from {old_vocab} to {new_vocab_size} tokens")
    else:
        print("No vocab growth detected; skipping embedding/output resize.")
        new_vocab_size = old_vocab
        opt.vocab_size = new_vocab_size
        opt.pad_token_id = tokenizer.pad_token_id
        model.register_buffer('indices', torch.arange(new_vocab_size, dtype=torch.long))

    print("=== Resize phase complete ===\n")

    # Move model to device
    print("Moving model to device...")
    model.to(opt.device)
    if opt.device.type == "cuda":
        torch.cuda.empty_cache()

    # ======================================================================
    #                          DATA CACHING PART
    # ======================================================================

    # Where to store tokenized data (per experiment)
    # opt.dir_name is something like "saved/model//"
    data_cache_dir = "saved/tokenizer"
    os.makedirs(data_cache_dir, exist_ok=True)

    train_pkl = os.path.join(data_cache_dir, "wiki.train.pkl")
    valid_pkl = os.path.join(data_cache_dir, "wiki.valid.pkl")
    test_pkl  = os.path.join(data_cache_dir, "wiki.test.pkl")
    meta_path = os.path.join(data_cache_dir, "tokenizer_meta.json")

    # Hash tokenizer vocab so we can detect changes
    vocab = tokenizer.get_vocab()
    vocab_hash = hashlib.md5(
        json.dumps(sorted(vocab.items())).encode("utf-8")
    ).hexdigest()

    force_retokenize = getattr(opt, "retokenize", False)

    # Check if cached data exists and matches current tokenizer
    files_exist = (
        os.path.exists(train_pkl) and
        os.path.exists(valid_pkl) and
        os.path.exists(test_pkl) and
        os.path.exists(meta_path)
    )

    use_cache = False
    if files_exist and not force_retokenize:
        try:
            with open(meta_path, "r") as f:
                meta = json.load(f)
            if meta.get("vocab_hash") == vocab_hash:
                use_cache = True
                print("Found matching tokenized data cache for current tokenizer.")
            else:
                print("Tokenizer changed since cached data was created; re-tokenizing.")
        except Exception as e:
            print(f"Could not read tokenizer metadata ({e}); re-tokenizing.")
    elif force_retokenize:
        print("Forced re-tokenization requested via opt.retokenize=True.")

    # --- Use cached data if valid ---
    if use_cache:
        print("Loading pre-tokenized data from cache...")
        try:
            with open(train_pkl, 'rb') as f: opt.train = pickle.load(f)
            with open(valid_pkl, 'rb') as f: opt.valid = pickle.load(f)
            with open(test_pkl,  'rb') as f: opt.test  = pickle.load(f)
        except Exception as e:
            print(f"Error loading cached pickles: {e}. Falling back to re-tokenization.")
            use_cache = False

    # --- Otherwise, tokenize from text and save ---
    if not use_cache:
        print("Tokenizing data from wiki.*.txt ...")
        try:
            opt.train = read_corpus('wiki.train.txt', tokenizer)
            opt.valid = read_corpus('wiki.valid.txt', tokenizer)
            opt.test  = read_corpus('wiki.test.txt', tokenizer)

            # Save tokenized data
            with open(train_pkl, 'wb') as f: pickle.dump(opt.train, f)
            with open(valid_pkl, 'wb') as f: pickle.dump(opt.valid, f)
            with open(test_pkl,  'wb') as f: pickle.dump(opt.test,  f)

            # Save tokenizer metadata (for detecting changes)
            with open(meta_path, "w") as f:
                json.dump({"vocab_hash": vocab_hash}, f)

            print(f"Data tokenized and cached in {data_cache_dir}.")
        except FileNotFoundError:
            print("Error: Source text files (wiki.*.txt) not found for tokenization.")
            return False
        except Exception as e:
            print(f"Unexpected error during tokenization: {e}")
            return False

    return True


# Main

In [ ]:
def main(opt):

    opt.device = torch.device("cpu") if getattr(opt, "no_cuda", False) \
        else (torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu"))

    # Memory optimization settings
    if opt.device.type == "cuda":
        torch.cuda.empty_cache()
        # Enable TF32 for better performance on Ampere GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    #  logging
    time_name = time.strftime("%y%m%d_%H%M%S")
    opt.time_name = time_name
    full_dir_path = f"saved/{opt.dir_name}"
    os.makedirs(full_dir_path, exist_ok=True)
    opt.dir_name = f"{full_dir_path}//" # Ensure trailing slashes for path concatenation
    opt.log_file = opt.dir_name + "log.txt"
    OutText(opt.log_file, opt); OutText(str(opt), opt)


    # force_retokenize can be a CLI flag or an opt attribute; default False
    force_retokenize = getattr(opt, "force_retokenize", False)

    tokenizer = load_or_build_tokenizer(opt, force_retokenize=force_retokenize)

    base_vocab_size = len(tokenizer)
    print(f"Vocabulary size (cached tokenizer): {base_vocab_size}")


    # Removed unconditional opt.retokenize = True
    # It is now handled gracefully in setup_model_and_data

    #  find & read checkpoint first (to infer sizes)
    ckpt_path = None
    if getattr(opt, "loadname", None):
        p = os.path.join(opt.loadname, "model_weights")
        if os.path.isfile(p):
            ckpt_path = p

    state = None
    if ckpt_path is not None:
        print("loading pretrained weights from: ", ckpt_path)
        try:
            state = torch.load(ckpt_path, map_location="cpu")
            if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
                state = state["state_dict"]

            # Initialize heads_inferred to a default value before attempting to infer from checkpoint
            heads_inferred = opt.heads

            # infer sizes from checkpoint
            emb_key = None
            for k in ["decoder.embed.embed.weight", "tok_embed.weight", "embedding.weight", "embed_tokens.weight"]:
                if k in state:
                    emb_key = k; break
            if emb_key is None:
                print("Warning: Could not find embedding weight in checkpoint. Model parameters might not be inferred.")
            else: # Only infer if emb_key was found
                vocab_old, d_model_old = state[emb_key].shape

                # count layers
                n_layers = -1
                # Corrected regex pattern
                pat = re.compile(r"decoder\.layers\.(\d+)\.")
                for k in state.keys():
                    m = pat.match(k)
                    if m:
                        n_layers = max(n_layers, int(m.group(1)))
                if n_layers < 0:
                    print("Warning: Could not infer number of layers from checkpoint. Using default opt.n_layers.")
                    n_layers = opt.n_layers # Fallback to default
                else:
                    n_layers = n_layers + 1

                # ff dim
                ff_dim = state.get("decoder.layers.0.ff.linear_1.weight", None)
                ff_dim = ff_dim.shape[0] if ff_dim is not None else 4 * d_model_old

                # max_len (positional encoding size)
                pe = state.get("decoder.pe.pe", None)
                max_len = pe.shape[1] if pe is not None else getattr(opt, "seqlen", 4096) # Use opt.seqlen as a default for max_len if not in state

                # GPT-2 base uses 12 heads with d_model=768
                heads = 12

                print(f"Inferred from ckpt: d_model={d_model_old}, n_layers={n_layers}, ff_dim={ff_dim}, "
                      f"heads={heads}, max_len={max_len}, vocab={vocab_old}")

                # override opt to match ckpt
                opt.d_model = d_model_old
                opt.n_layers = n_layers
                # if hasattr(opt, "d_ff"): opt.d_ff = ff_dim # d_ff not directly in Args, but could be passed
                opt.heads = heads
                opt.seqlen = max_len # Use inferred max_len for opt.seqlen (model capacity, not training seqlen)
                print(f"Note: opt.seqlen={opt.seqlen} is model capacity. Actual training will use data's max_length.")

        except RuntimeError as e:
            print(f"Error loading checkpoint '{ckpt_path}': {e}")
            print("This often happens if the checkpoint was saved with a different PyTorch version or is corrupted.")
            print("Proceeding with random initialization. If you have access to the original environment, consider re-saving the checkpoint with your current PyTorch version.")
            state = None # Fallback to random initialization
        except Exception as e:
            print(f"An unexpected error occurred while loading checkpoint '{ckpt_path}': {e}")
            print("Proceeding with random initialization.")
            state = None # Fallback to random initialization


    # Build model with BASE vocab size
    opt.vocab_size = base_vocab_size
    opt.pad_token_id = tokenizer.pad_token_id
    model = get_modelGPT(opt, opt.vocab_size)
    # Load checkpoint (sizes should match now!)
    if state is not None:
        load_ckpt_allowing_pad_growth(model, state, verbose=True)

    model = apply_lora(model, r=4, modules=["q_linear", "k_linear", "v_linear", "out"])


    success = setup_model_and_data(model, tokenizer, opt)
    if not success:
        return

    pad_id = tokenizer.pad_token_id

    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum(int(np.prod(p.size())) for p in model_parameters)
    OutText(f'total params: {params}', opt)

    opt.optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, betas=(0.9, 0.98), eps=1e-9)
    if getattr(opt, "SGDR", False):
        opt.sched = CosineWithRestarts(opt.optimizer, T_max=len(opt.train))

    if getattr(opt, "savename", None):
        os.makedirs(opt.savename, exist_ok=True)

    opt.src_pad = tokenizer.pad_token_id # Set src_pad as well for completeness
    opt.trg_pad = tokenizer.pad_token_id

    print(model.named_modules())

    #  Zero-Shot Evaluation (Added for subtask)
    print("\n### Running Zero-Shot Evaluation (before fine-tuning) ###")
    testmodel_obqa(model, opt, tokenizer, print_examples=True)
    print("### Zero-Shot Evaluation Complete ###\n")

    #  run finetuning
    finetune_obqa(model, opt, tokenizer)
    testmodel_obqa(model, opt, tokenizer, print_examples=True)

In [ ]:
import time

start_time = time.time()
main(opt)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total execution time for main: {elapsed_time:.2f} seconds")

/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


saved/model//log.txt
Loading cached tokenizer from saved/tokenizer
Vocabulary size (cached tokenizer): 50262

=== Adding special tokens and resizing model (idempotent) ===
Added 5 new special tokens
Current additional special tokens: ['[START]', '[ANSWER]', '[CLS]', '[SEP]', '[END]']
Tokenizer vocab size: 50267, model vocab size: 50262
Resizing embeddings and output layer (memory-efficient)...
Model resized from 50262 to 50267 tokens
=== Resize phase complete ===

Moving model to device...
Found matching tokenized data cache for current tokenizer.
Loading pre-tokenized data from cache...
total params: 39104088
<generator object Module.named_modules at 0x7d2f228c3230>

### Running Zero-Shot Evaluation (before fine-tuning) ###
validating....
0 {'fact': 'using less resources usually causes money to be saved', 'stem': 'A person wants to start saving money so that they can afford a nice vacation at the end of the year. After looking over their budget and expenses, they decide the best way t